## **Library**

In [19]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans

from catboost import CatBoostClassifier
import seaborn as sns

import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_classif

In [20]:
df = pd.read_csv("playground-series-s6e6/train.csv")

baris, kolom = df.shape
print(f"Baris: {baris}, Kolom: {kolom}")
df.head()

Baris: 577347, Kolom: 12


,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY


## **Data Cleaning**

In [21]:
missing_data = df.isnull().sum()
missing_data = missing_data.sort_values(ascending= False)
missing_data

id                   0
alpha                0
delta                0
u                    0
g                    0
r                    0
i                    0
z                    0
redshift             0
spectral_type        0
galaxy_population    0
class                0
dtype: int64

Data tidak ada yang hilang

In [22]:
y = df["class"]
X = df.drop(["class"], axis=1)
cat_col = ['spectral_type', 'galaxy_population']

## **Mutual Information**

In [23]:
# Label encoding for categoricals
for colname in X.select_dtypes("object"):
    X[colname], _ = X[colname].factorize()
diskrit = X.dtypes == int

In [24]:
def make_mi_scores(X, y, discrete_features):
    mi_scores = mutual_info_classif(X, y, discrete_features=discrete_features)
    mi_scores = pd.Series(mi_scores, name="MI Scores", index=X.columns)
    mi_scores = mi_scores.sort_values(ascending=False)
    return mi_scores

mi_scores = make_mi_scores(X, y, diskrit)
mi_scores

id                   0.879847
redshift             0.514987
spectral_type        0.299984
z                    0.211515
galaxy_population    0.192690
u                    0.178885
g                    0.174582
i                    0.158252
alpha                0.139271
delta                0.130430
r                    0.097934
Name: MI Scores, dtype: float64

In [25]:
model = CatBoostClassifier(
    iterations=1000, 
    learning_rate= 0.05,
    depth = 8,
    loss_function= 'MultiClass',
    eval_metric= 'TotalF1',
    cat_features= cat_col,
    random_seed= 42)

## **Feature Engineering**

In [26]:
#seberapa  dekat ke infrared
df["u-g"] = df["u"] - df["g"]
df["g-r"] = df["g"] - df["r"]
df["r-i"] = df["r"] - df["i"]
df["i-z"] = df["i"] - df["z"]

fe = ["u-g", 'g-r','r-i', 'i-z']

### **MI Score after Feature Engineering**

In [27]:
y = df["class"]
X = df.drop(["class", "alpha", "delta"], axis=1)

# Label encoding for categoricals
for colname in X.select_dtypes("object"):
    X[colname], _ = X[colname].factorize()
diskrit = X.dtypes == int

def make_mi_scores(X, y, discrete_features):
    mi_scores = mutual_info_classif(X, y, discrete_features=discrete_features)
    mi_scores = pd.Series(mi_scores, name="MI Scores", index=X.columns)
    mi_scores = mi_scores.sort_values(ascending=False)
    return mi_scores

mi_scores = make_mi_scores(X, y, diskrit)
mi_scores

id                   0.879847
redshift             0.514988
g-r                  0.321153
spectral_type        0.299984
r-i                  0.218323
z                    0.211509
galaxy_population    0.192690
u                    0.178881
g                    0.174593
u-g                  0.171347
i                    0.158246
r                    0.097934
i-z                  0.074761
Name: MI Scores, dtype: float64

### **Clustering**

In [28]:
filter_col = ['u', 'g', 'r', 'i', 'z']

kmeans = KMeans(n_clusters= 3)

X["Cluster"] = kmeans.fit_predict(df[filter_col])
X["Cluster"] = X["Cluster"].astype("category")

X.head()

,id,u,g,r,i,z,redshift,spectral_type,galaxy_population,u-g,g-r,r-i,i-z,Cluster
0,0,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,0,0,3.576564,1.537632,1.100813,0.636056,1
1,1,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,0,0,1.691447,1.499854,0.361141,0.439634,2
2,2,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,1,1,-0.043925,-0.092712,0.589211,0.025263,0
3,3,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,0,0,2.254320,2.032982,0.652096,0.450706,0
4,4,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,0,0,2.231478,1.237231,0.335002,0.283262,0
